In [89]:
import numpy as np
import gym

In [90]:
def policy_evaluation(env, policy, gamma=0.99, theta=1e-5):
    n_states = env.observation_space.n
    V = np.zeros(n_states)

    while True:
        delta = 0
        for s in range(n_states):
            v = 0
            a = policy[s]
            for prob, next_state, reward, done in env.P[s][a]:
                v += prob * (reward + gamma * V[next_state] * (not done))
            delta = max(delta, abs(v - V[s]))
            V[s] = v
        if delta < theta:
            break
    return V

def policy_improvement(env, V, gamma=0.99):
    policy = np.zeros(env.observation_space.n, dtype=int)
    for s in range(env.observation_space.n):
        action_values = []
        for a in range(env.action_space.n):
            q = 0
            for prob, next_state, reward, done in env.P[s][a]:
                q += prob * (reward + gamma * V[next_state] * (not done))
            action_values.append(q)
        policy[s] = np.argmax(action_values)
    return policy

def policy_iteration(env, gamma=0.8, theta=1e-6):
    n_states = env.observation_space.n
    policy = np.zeros(n_states, dtype=int)  # Initialize with zeros
    stable = False
    iteration = 0
    while not stable:
        V = policy_evaluation(env, policy, gamma, theta)
        new_policy = policy_improvement(env, V, gamma)

        if np.array_equal(policy, new_policy):
            stable = True
        else:
            policy = new_policy
        iteration += 1

    print(f"Policy Iteration converged in {iteration} iterations.")
    return policy, V

In [92]:
def evaluate_policy(env, policy, episodes=1):
    total_rewards = []
    for _ in range(episodes):
        state = env.reset()[0]
        done = False
        total_reward = 0
        while not done:
            action = policy[state]
            state, reward, done, _, info = env.step(action)
            total_reward += reward
        total_rewards.append(total_reward)
    avg_reward = np.mean(total_rewards)
    print(f"Average reward over {episodes} episodes: {avg_reward}")
    return avg_reward

In [91]:
def print_policy(policy, shape=(4, 12)):
    symbols = {
        0: '↑',  # UP
        1: '→',  # RIGHT
        2: '↓',  # DOWN
        3: '←'   # LEFT
    }
    
    grid = np.array([symbols[a] for a in policy]).reshape(shape)

    print("Policy Grid:")
    for row in grid:
        print(" ".join(row))

In [97]:
env = gym.make("CliffWalking-v0")
optimal_policy, optimal_values = policy_iteration(env)
print_policy(optimal_policy)
#Optionally evaluate the learned policy
evaluate_policy(env, optimal_policy)

Policy Iteration converged in 15 iterations.
Policy Grid:
→ → → → → → → → → → → ↓
→ → → → → → → → → → → ↓
→ → → → → → → → → → → ↓
↑ ↑ ↑ ↑ ↑ ↑ ↑ ↑ ↑ ↑ → →
Average reward over 1 episodes: -13.0


-13.0